## 05 · Multi-Agent Systems

### Why Multi-Agent?

A single agent trying to do everything becomes:
- Hard to maintain
- Hard to debug
- Slow (too many tools confuse the LLM)

**Multi-Agent** splits work across **specialized agents**:

```
             User Request
                  │
           ┌──────▼──────┐
           │  SUPERVISOR │  ← Routes to the right agent
           └──────┬──────┘
          ┌───────┼────────┐
    ┌─────▼──┐ ┌──▼────┐ ┌─▼──────┐
    │Research│ │  Math │ │ Sales  │
    │ Agent  │ │ Agent │ │ Agent  │
    └────────┘ └───────┘ └────────┘
```

### Patterns Covered
| Pattern | Description |
|---------|-------------|
| **Agent as Tool** | One agent calls another agent as a tool |
| **Supervisor Router** | Supervisor decides which specialist to call |
| **Sequential Pipeline** | Agent A output feeds Agent B |

In [1]:
import os
from load_dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
print("LLM ready")

c:\Users\arunm\OneDrive\Desktop\LangChain_Tutorial\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


LLM ready


## Pattern 1 — Agent as a Tool
Build **specialist agents** and expose them as tools to a supervisor.

We'll create:
- **Research Agent** — search + Wikipedia
- **Math Agent** — calculations
- **Supervisor Agent** — routes user queries to the right specialist

In [2]:
# ── RESEARCH AGENT ─────────────────────────────────────────
search_tool = DuckDuckGoSearchRun(description="Web search for recent news and events")
wiki_tool = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=2),
    description="Wikipedia for factual and historical information",
)

research_agent = create_agent(
    model=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    tools=[search_tool, wiki_tool],
)

# ── MATH AGENT ─────────────────────────────────────────────
@tool
def calculator(expression: str) -> str:
    """Evaluate a math expression. Input must be a Python-safe arithmetic expression."""
    try:
        allowed = {"__builtins__": {}, "pow": pow, "abs": abs, "round": round}
        return str(eval(expression, allowed))  # noqa: S307
    except Exception as e:
        return f"Error: {e}"

@tool
def unit_converter(query: str) -> str:
    """Convert units. Supports km↔miles, kg↔lbs, Celsius↔Fahrenheit.
    Input format: '<value> <from_unit> to <to_unit>'. Example: '100 km to miles'"""
    import re
    q = query.lower()
    match = re.search(r"([\d.]+)\s*(\w+)\s*to\s*(\w+)", q)
    if not match:
        return "Invalid format. Use: '100 km to miles'"
    val, from_u, to_u = float(match.group(1)), match.group(2), match.group(3)
    conversions = {
        ("km", "miles"): val * 0.621371,
        ("miles", "km"): val * 1.60934,
        ("kg", "lbs"): val * 2.20462,
        ("lbs", "kg"): val / 2.20462,
        ("celsius", "fahrenheit"): val * 9/5 + 32,
        ("fahrenheit", "celsius"): (val - 32) * 5/9,
    }
    result = conversions.get((from_u, to_u))
    if result is None:
        return f"Conversion from {from_u} to {to_u} not supported"
    return f"{val} {from_u} = {result:.4f} {to_u}"

math_agent = create_agent(
    model=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    tools=[calculator, unit_converter],
)

print("Specialist agents created:")
print("  • Research Agent (search + wiki)")
print("  • Math Agent (calculator + unit converter)")

Specialist agents created:
  • Research Agent (search + wiki)
  • Math Agent (calculator + unit converter)


In [3]:
# Wrap each specialist agent as a tool for the supervisor

@tool
def research_agent_tool(query: str) -> str:
    """Use this for research questions, factual lookups, news, and general knowledge.
    Searches the web and Wikipedia to find accurate information."""
    result = research_agent.invoke({"messages": [("user", query)]})
    return result["messages"][-1].content

@tool
def math_agent_tool(query: str) -> str:
    """Use this for mathematical calculations, arithmetic, unit conversions,
    and any problem requiring numbers or formulas."""
    result = math_agent.invoke({"messages": [("user", query)]})
    return result["messages"][-1].content

print("Specialist agents wrapped as tools ✓")

Specialist agents wrapped as tools ✓


In [4]:
# Build the Supervisor Agent
from langchain_core.messages import SystemMessage

supervisor_system = """
You are a supervisor agent that routes user queries to specialist sub-agents.

Available specialists:
- research_agent_tool: For factual questions, history, news, general knowledge
- math_agent_tool: For calculations, arithmetic, unit conversions, numbers

Route each query to the most appropriate specialist and return their answer.
For complex queries, you may call multiple specialists and synthesize the results.
"""

supervisor_agent = create_agent(
    model=llm,
    tools=[research_agent_tool, math_agent_tool],
)

print("Supervisor Agent created ✓")

Supervisor Agent created ✓


In [5]:
# Test: Research query
query = "Who invented the internet?"
print(f"Query: {query}")
print("=" * 60)

events = supervisor_agent.stream(
    {"messages": [("user", query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

Query: Who invented the internet?
================================ Human Message =================================

Who invented the internet?
================================== Ai Message ==================================
Tool Calls:
  research_agent_tool (call_TUsuGOCr9H3gbpREPGbJmT8a)
 Call ID: call_TUsuGOCr9H3gbpREPGbJmT8a
  Args:
    query: Who invented the internet?
================================= Tool Message =================================
Name: research_agent_tool

The Internet was not invented by a single person but rather originated from the collaborative efforts of multiple scientists and engineers. Key figures in its development include:

1. **J. C. R. Licklider** - He envisioned a universal network at the U.S. Department of Defense's ARPA (Advanced Research Projects Agency).
2. **Paul Baran** - Proposed a distributed network based on message blocks in the early 1960s.
3. **Donald Davies** - Conceived packet switching in 1965 and proposed a national commercial data ne

In [6]:
# Test: Math query
query = "Convert 500 kilometers to miles and then multiply by 2."
print(f"Query: {query}")
print("=" * 60)

events = supervisor_agent.stream(
    {"messages": [("user", query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

Query: Convert 500 kilometers to miles and then multiply by 2.
================================ Human Message =================================

Convert 500 kilometers to miles and then multiply by 2.
================================== Ai Message ==================================
Tool Calls:
  math_agent_tool (call_HkfCYQEnJNq2G8HOzQGWXioz)
 Call ID: call_HkfCYQEnJNq2G8HOzQGWXioz
  Args:
    query: 500 kilometers to miles
  math_agent_tool (call_s9ZekdL61sjStU5RCqOV9etb)
 Call ID: call_s9ZekdL61sjStU5RCqOV9etb
  Args:
    query: 500 kilometers * 2
================================= Tool Message =================================
Name: math_agent_tool

The result of \( 500 \) kilometers multiplied by \( 2 \) is \( 1000 \) kilometers.
================================== Ai Message ==================================

500 kilometers is approximately 310.69 miles. When you multiply 500 kilometers by 2, you get 1000 kilometers.


In [7]:
# Test: Mixed query — needs BOTH agents
query = "What is the distance from Mumbai to Delhi in km? Convert that to miles."
print(f"Query: {query}")
print("=" * 60)

events = supervisor_agent.stream(
    {"messages": [("user", query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

Query: What is the distance from Mumbai to Delhi in km? Convert that to miles.
================================ Human Message =================================

What is the distance from Mumbai to Delhi in km? Convert that to miles.
================================== Ai Message ==================================
Tool Calls:
  research_agent_tool (call_QckTgZeT4Hm5SZpT6OUWMQZ3)
 Call ID: call_QckTgZeT4Hm5SZpT6OUWMQZ3
  Args:
    query: distance from Mumbai to Delhi in km
  math_agent_tool (call_C2ePKymQyFOo8VWCNQeoHbGI)
 Call ID: call_C2ePKymQyFOo8VWCNQeoHbGI
  Args:
    query: convert km to miles for 1 km
================================= Tool Message =================================
Name: math_agent_tool

1 kilometer is equal to approximately 0.6214 miles.
================================== Ai Message ==================================
Tool Calls:
  math_agent_tool (call_FQ8XNpcQSrfF5GhPmct4sssF)
 Call ID: call_FQ8XNpcQSrfF5GhPmct4sssF
  Args:
    query: convert 1400 km to miles
====

## Pattern 2 — Sequential Pipeline
Agent A researches a topic → Agent B summarizes the findings into structured output.

In [8]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Step 1: Research Agent gathers raw information
def research_step(topic: str) -> str:
    result = research_agent.invoke({"messages": [("user", f"Research this topic thoroughly: {topic}")]})
    return result["messages"][-1].content

# Step 2: Summarizer LLM chain formats into bullet points
summarizer_prompt = ChatPromptTemplate([
    ("system", "You are an expert at creating concise, presentation-ready summaries."),
    ("user", "Format this research into 5 bullet points for a slide:\n\n{research}")
])
summarizer = summarizer_prompt | llm | StrOutputParser()

# Run the pipeline
topic = "Large Language Models"
print(f"Topic: {topic}")
print("=" * 60)

print("Step 1: Researching...")
raw_research = research_step(topic)
print(f"Research gathered: {len(raw_research)} characters\n")

print("Step 2: Summarizing...")
summary = summarizer.invoke({"research": raw_research})
print("\nPresentation Summary:")
print(summary)

Topic: Large Language Models
Step 1: Researching...
Research gathered: 2945 characters

Step 2: Summarizing...

Presentation Summary:
- **Definition & Purpose**: LLMs are advanced models for natural language processing with billions to trillions of parameters, utilizing self-supervised learning for versatile language tasks like generation and translation.

- **Architecture**: Primarily based on the transformer architecture, LLMs employ self-attention mechanisms for efficient processing, leading to powerful models like GPT and BERT with capabilities such as few-shot learning.

- **Training Techniques**: Techniques include fine-tuning with reinforcement learning from human feedback (RLHF) and comprehensive multi-task benchmarking, although there's concern over potential overfitting to benchmarks.

- **Capabilities**: LLMs excel in various applications including conversational agents, code generation, knowledge retrieval, and automated reasoning, revolutionizing fields that once relied on